In [29]:
# import modules
# note: need to use env conda potentials
import pickle
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from xarray import DataTree

import matplotlib.pyplot as plt
import seaborn as sns

In [30]:
frame_time = 0.006715 #s

In [31]:
root_path = Path.cwd() / Path('data_for_checking_lm_fits')

In [32]:
dt = DataTree(name="Data from DHM", children={'3D': DataTree(name='3D'), '2D': DataTree(name='2D')})

for path in root_path.glob('?D*/[0-9]*'):
    # First loop through the subdirectories and find all the concentrations 
    concentration = path.parts[-1]
    dimensionality = path.parts[-2].split('_', 1)[0]
    # add each concentration as a node to the data tree
    dt[dimensionality][concentration] = DataTree(name=concentration)

    # now look up the time values for each run and use as 
    # coordinate in a Dataset
    run = '0'
    times = np.arange(0,100)*frame_time
    ds = xr.Dataset(coords={'time':times})
    dt[dimensionality][concentration][run] = DataTree(ds)
    # now add the other parameters to the dataset
    for filename in path.glob('*[0-9].p'):
        parameter_set = filename.parts[-1].rsplit('_', 1)[0]
        run = filename.parts[-1].split('_')[-1].split('.')[0]
        file = open(filename, 'rb')
        data = pickle.load(file)
        file.close()
        # for files containing multiple values (for example, gap distances and uncertainties), 
        # we'll store the the names of the values as a new coordinate in a DataArray
        if isinstance(data, dict):
            try:
                df = pd.DataFrame.from_dict(data)
                df.index = dt[dimensionality][concentration][run].coords['time']
                #print(df)
                da = xr.DataArray(df).rename({'dim_0' : 'time', 'dim_1' : parameter_set + '_vars'})
                dt[dimensionality][concentration][run][parameter_set] = da
            except KeyError as error:
                print("Missing time array.")
                print("file:", filename.relative_to(root_path))
                print("\n")
            except ValueError as error:
                print("Mismatch between length of time array and length of data.")
                print("file:", filename.relative_to(root_path))
                print("dimensions of time:", dt[dimensionality][concentration][run].coords['time'].shape)
                print("dimensions of gap_mcmc_unc:", df.shape)
                print("\n")
        # handle the r1, r2, n1, n2 variables
        else:
            try:
                if(parameter_set != 'time'):
                    da = xr.DataArray(data, coords={'time': dt[dimensionality][concentration][run].coords['time']})
                    dt[dimensionality][concentration][run][parameter_set] = da
                else:
                    # don't add the time array to the dataarray (it's a coord, not a variable)
                    pass
            except KeyError as error:
                print("Missing time array.")
                print("file:", filename.relative_to(root_path))
                print("\n")

In [33]:
example_data = dt['3D']['0.0875']['0'].dataset
example_data

<xarray.DatasetView> Size: 9kB
Dimensions:   (time: 100, gap_vars: 3)
Coordinates:
  * time      (time) float64 800B 0.0 0.006715 0.01343 ... 0.6514 0.6581 0.6648
  * gap_vars  (gap_vars) object 24B 'gap' 'gap_plus' 'gap_minus'
Data variables:
    n2        (time) float64 800B 1.599 1.599 1.599 1.599 ... 1.599 1.599 1.599
    n1        (time) float64 800B 1.584 1.584 1.584 1.584 ... 1.584 1.584 1.584
    gap       (time, gap_vars) float64 2kB 0.09994 1.533e-11 ... 0.0001322
    theta     (time) float64 800B 2.652 2.635 2.615 2.622 ... 2.274 2.239 2.243
    r1        (time) float64 800B 0.6476 0.6513 0.6538 ... 0.6625 0.6653 0.6622
    r2        (time) float64 800B 0.6698 0.6662 0.6652 ... 0.646 0.6344 0.6333
    phi       (time) float64 800B 0.05016 0.05093 0.0745 ... -1.558 -1.521
    z         (time) float64 800B 21.0 21.0 21.0 21.01 ... 20.84 20.82 20.82

In [34]:
phi = dt['3D']['0.0875']['0']['phi']

In [35]:
print(phi)

<xarray.DataArray 'phi' (time: 100)> Size: 800B
array([ 5.01608983e-02,  5.09255275e-02,  7.44987990e-02,  4.93210036e-02,
        1.14949140e-01,  2.22129736e-01,  1.85043145e-01,  2.61688759e-01,
        2.55958683e-01,  1.81208340e-01,  1.14149961e-03,  2.61227645e-02,
        2.61163990e-02,  5.03210484e-02,  2.11729564e-01,  1.54959415e-01,
        4.75811885e-02,  6.76959932e-02,  1.67026841e-01,  1.59510810e-01,
       -1.20667617e-01, -9.86639837e-02, -1.72529457e-01, -2.72977030e-01,
       -3.12731630e-01, -4.33205519e-01, -5.00125797e-01, -5.04964598e-01,
       -5.04846786e-01, -4.80845797e-01, -4.79655133e-01, -4.43439320e-01,
       -4.31639342e-01, -3.72936895e-01, -4.12819868e-01, -3.04058463e-01,
       -3.71898132e-01, -4.21038788e-01, -5.84231755e-01, -3.67529062e-01,
       -3.63806695e-01, -2.97630226e-01, -3.60513306e-01, -4.07719947e-01,
       -5.38062691e-01, -4.63540118e-01, -5.26817617e-01, -5.85743510e-01,
       -6.46164593e-01, -5.96275147e-01, -6.18243862

In [36]:
r1 = dt['3D']['0.0875']['0']['r1']
print(r1)

<xarray.DataArray 'r1' (time: 100)> Size: 800B
array([0.64757287, 0.65129239, 0.65381064, 0.65260674, 0.65135628,
       0.654577  , 0.64827286, 0.64922442, 0.65508719, 0.64583473,
       0.64833221, 0.64534467, 0.64864834, 0.65276786, 0.65259684,
       0.65399424, 0.64818741, 0.64234544, 0.65095447, 0.64354388,
       0.64389473, 0.64371357, 0.64702804, 0.65043517, 0.64561952,
       0.63581971, 0.63771406, 0.63720564, 0.63532595, 0.64167392,
       0.67707831, 0.65938179, 0.65857522, 0.64971904, 0.64871217,
       0.65018052, 0.64587549, 0.64296906, 0.6368435 , 0.63742403,
       0.63764474, 0.64398505, 0.64626944, 0.64941875, 0.64051837,
       0.65073387, 0.65413537, 0.65110312, 0.66006693, 0.64957154,
       0.65625052, 0.65840696, 0.65844129, 0.66033377, 0.6640664 ,
       0.65686525, 0.65698664, 0.65706585, 0.65380035, 0.65609942,
       0.6564115 , 0.65508413, 0.65702109, 0.65899871, 0.65803906,
       0.66071557, 0.66005965, 0.6586391 , 0.65881967, 0.65896097,
       0.657947